## Test PyTorch

In [ ]:
import torch


def print_gpu_info():

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        print("Current device:", torch.cuda.current_device())
        print("GPU name:", torch.cuda.get_device_name(0))

        x = torch.randn(1000, 1000, device="cuda")
        y = torch.randn(1000, 1000, device="cuda")
        z = x @ y

        torch.cuda.synchronize()
        print("Success! Tensor is on:", z.device)
    else:
        print("No GPU available.")

print_gpu_info()

## Setup logging

In [ ]:
import logging, os, sys

from dask.typing import Key
from distributed import Worker, WorkerPlugin


def configure_logging(config_file, verbose, logger_name=None):
    if config_file and os.path.exists(config_file):
        print(f'Configure logging using {config_file}, logger name: {logger_name}')
        logging.config.fileConfig(config_file)
    else:
        print(f'Configure logging using basic config - verbose: {verbose}, logger name: {logger_name}')
        log_format = '%(asctime)s - %(threadName)s:%(name)s - %(levelname)s - %(message)s'
        log_level = logging.DEBUG if verbose else logging.INFO
        logging.basicConfig(level=log_level,
                            format=log_format,
                            datefmt='%Y-%m-%d %H:%M:%S',
                            handlers=[
                                logging.StreamHandler(stream=sys.stdout)
                            ])
    return logging.getLogger(logger_name)


class ConfigureWorkerPlugin(WorkerPlugin):
    def __init__(self, models_dir, logging_config, verbose):
        self.models_dir = models_dir
        self.logging_config = logging_config
        self.verbose = verbose
        self.logger = None

    def setup(self, worker: Worker):
        self.logger = configure_logging(self.logging_config, self.verbose,
                                        logger_name='dask_worker')
        if self.models_dir:
            self.logger.info(f'Set cellpose models path: {self.models_dir}')
            os.environ['CELLPOSE_LOCAL_MODELS_PATH'] = self.models_dir

    def teardown(self, worker: Worker):
        pass

    def transition(self, key: Key, start: str, finish: str, **kwargs):
        pass

    def release_key(self, key: str, state: str, cause: str | None, reason: None, report: bool):
        pass


log_config_file = ''

logger = configure_logging(log_config_file, True)


In [ ]:
import numpy as np
import numcodecs as zv2codecs
import zarr
import zarr.codecs as zv3codecs


def open_zarr(img_container_path, img_subpath, mode='r'):
    img_container = zarr.open(store=img_container_path, mode=mode)
    if img_subpath:
        return img_container[img_subpath]
    else:
        return img_container


def create_zarr(zarr_container_path, dataset_subpath,
                dataset_3dshape,
                dtype='uint32',
                chunks=(64,64,64),
                sharding_factors=(4,4,4),
                zarr_format=2):

    logger.info(f'Create zarr container {zarr_container_path}:{dataset_subpath}')
    zarr_container = zarr.open_group(
        store=zarr_container_path,
        mode='w',
        zarr_format=zarr_format,
    )

    if zarr_format == 2:
        zstd_codec = zv2codecs.get_codec({'id': 'zstd', 'level': 5})
        compressors = {'compressors': zstd_codec}
        chunk_key_separator = {'name': 'v2', 'separator': '/'}
        sharding_args = {}
    else:
        zstd_codec = zv3codecs.ZstdCodec(level=5)
        compressors = {'compressors': (zstd_codec,)}
        chunk_key_separator = None
        shard_shape = tuple(int(x) for x in np.asarray(chunks) * sharding_factors)
        sharding_args = {'shards': shard_shape}
        logger.info(f'Use zarrv3 sharding: {sharding_args}')

    return zarr_container.create_array(
        name=dataset_subpath,
        shape=dataset_3dshape,
        chunks=chunks,
        dtype=dtype,
        chunk_key_encoding=chunk_key_separator,
        **compressors,
        **sharding_args,
    )

## Method to create the dask cluster

In [ ]:
from cellpose.contrib.distributed_segmentation import janeliaLSFCluster, myLocalCluster

def create_cluster(nworkers:int,
                   cluster_type:str='lsf',
                   config={},
                   worker_plugin=None,
                   **kwargs):
    if cluster_type == 'lsf':
        dask_cluster = janeliaLSFCluster(
            1, # ncpus per worker
            1, # min workers
            nworkers,
            config=config,
            **kwargs,
        )
    else:
        dask_cluster = myLocalCluster(
            1,
            n_workers=nworkers,
            threads_per_worker=1,
            processes=True,
            host="localhost",
            **kwargs,
        )
    if worker_plugin is not None:
        dask_cluster.client.register_plugin(worker_plugin, 'WorkerConfig')
    
    return dask_cluster


In [ ]:
import scipy
import traceback

from cellpose import utils

from dask.distributed import as_completed, Client
from segmentation.block_utils import get_block_crops, prepare_blocksize


def distributed_posteval(
    labels_zarr:zarr.Array,
    blocksize,
    dask_client: Client,
    mask=None,
    roi=None,
    input_zarr:zarr.Array=None,
    min_size_voxels=-1,
    input_channel=-1,
    input_high_percentile=98,
):
    labels_shape = labels_zarr.shape
    blocksize = prepare_blocksize(labels_shape, blocksize)
    block_indices, block_crops = get_block_crops(
        labels_shape, blocksize, None, mask, roi,
    )
    if len(block_indices) == 0:
        return labels_zarr

    futures = dask_client.map(
        _block_postprocess,
        block_indices,
        block_crops,
        labels_zarr=labels_zarr,
        input_zarr=input_zarr,
        min_size_voxels=min_size_voxels,
        input_channel=input_channel,
        input_high_percentile=input_high_percentile,
    )

    for f, r in as_completed(futures, with_results=True):
        if f.cancelled():
            tb = f.traceback()
            tb = f.traceback()
            stacktrace = ''.join(traceback.format_tb(tb))
            logger.error(f'Block error: {stacktrace}')
        else:
            bi, bc = r
            logger.info(f'Completed block {bi} at {bc}')
    
    return labels_zarr


def _block_postprocess(
    block_index,
    crop,
    labels_zarr,
    min_size_voxels=-1,
    input_zarr=None,
    input_channel=0,
    input_high_percentile=98,
):
    labels_block = labels_zarr[crop]
    logger.debug(f'Process block {block_index} at {crop}')
    if min_size_voxels > 0:
        filtered_labels_block = utils.fill_holes_and_remove_small_masks(labels_block, min_size=min_labels_voxels)
    else:
        filtered_labels_block = labels_block

    mask_ids = np.unique(filtered_labels_block)
    mask_ids = mask_ids[mask_ids != 0]

    # check the input is provided and there are foreground masks
    if input_zarr is not None and input_channel >= 0 and mask_ids.size > 0:
        logger.debug(f'Filter block {block_index} labels based on channel {input_channel} intensity')
        input_coord = (0,input_channel) + crop
        input_block = input_zarr[input_coord]
        input_avg_label_intensity = scipy.ndimage.mean(input_block, labels=filtered_labels_block, index=mask_ids)
        input_threshold = np.percentile(input_avg_label_intensity, input_high_percentile)
        dropped_mask_ids = mask_ids[input_avg_label_intensity < input_threshold]
        filtered_labels_block[np.isin(filtered_labels_block, dropped_mask_ids)] = 0
        labels_zarr[crop] = filtered_labels_block

    return block_index, crop
    

## Set input/output data

In [ ]:
test_input_img_zarr = open_zarr('/nrs/scicompsoft/goinac/multifish/tiny_fg/stitched.ome.zarr/', 't1/1')
test_input_img_shape = test_input_img_zarr.shape

test_working_dir = '/nrs/scicompsoft/goinac/multifish/tiny_fg/dseg'
test_labels_container = f'{test_working_dir}/labels.zarr'
test_labels_subpath = 'labels'
test_labels_zarr_format = 3

# Create the labels container
test_labels_zarr = create_zarr(
   test_labels_container,
   test_labels_subpath,
   test_input_img_shape[-3:],
   chunks=(64,64,64),
   zarr_format=test_labels_zarr_format)


## Invoke distributed segmentation

In [ ]:
from segmentation.distributed_cellpose import distributed_eval, distributed_merge


models_dir = ''
logging_config = ''
nworkers = 2

worker_plugin = ConfigureWorkerPlugin(models_dir, logging_config, True)

worker_directives = [
    "-P scicompsoft",
    "-gpu 'num=1'",
    "-q gpu_l4",
    "-J cellpose-worker",
]

dask_config = {}

lsf_worker_job_args = {
    'cluster_type': 'lsf',
    'job_extra_directives': worker_directives,
}

local_worker_job_args = {
    'cluster_type': 'local'
}

worker_job_args = {
    'worker_plugin': worker_plugin,
    'config': {},
    **lsf_worker_job_args,
}

blocksize = (128, 128, 128)
min_labels_voxels=1000
input_channel=0
input_high_percentile=98

with create_cluster(nworkers, **worker_job_args) as dask_cluster:
    logger.info('Run distributed eval')
    labels, _ = distributed_eval(
        test_input_img_zarr,
        0, # input_timeindex
        [1], # input_channels
        blocksize,
        test_working_dir,
        test_labels_zarr,
        dask_cluster.client,
        blockoverlaps=(),
        mask=None,
        roi=None,
        cellpose_model_args={
            'use_gpu': True,
            'gpu_device': '0',
            'pretrained_model': 'cpsam',
        },
        normalize_args={
            'normalize': True,
            'lowhigh': (1,99),
        },
        cellpose_eval_args={
            'do_3D': True,
            'diameter': 30,
            'min_size': 15,
            'max_size_fraction': 0.4,
            'cellprob_threshold': -8,
            'flow3D_smooth': 1,
            'batch_size': 8,
        },
        label_dist_th=1.0,        
        skip_merge_labels=True,
    )

    logger.info('Run post processing')
    labels = distributed_posteval(
        test_labels_zarr,
        blocksize,
        dask_cluster.client,
        mask=None,
        roi=None,
        input_zarr=test_input_img_zarr,
        min_size_voxels=min_labels_voxels,
        input_channel=input_channel,
        input_high_percentile=input_high_percentile,
    )

    logger.info('Run merge labels')
    labels, boxes = distributed_merge(
        labels,
        blocksize,
        labels,
        test_working_dir,
        dask_cluster.client,
        mask=None,
        roi=None,
        label_dist_th=1.0,
    )

logger.info(f'Completed distributed segmentation: {labels}')